# NFSP Poker Bot Training
## Neural Fictitious Self-Play for Short Deck Texas Hold'em

This notebook trains an NFSP agent to play Short Deck Poker (20 cards: 10-A x 4 suits).

**Features:**
- GPU-accelerated training
- Real-time metrics visualization
- Model checkpoints
- TensorBoard logging

## 1. Setup & Installation

In [ ]:
# Import required libraries
import torch
import numpy as np
import random
import time
import os
from collections import deque
from itertools import combinations
import matplotlib.pyplot as plt
from IPython.display import clear_output

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Configuration

In [ ]:
"""
config.py - Hyperparameters for NFSP Poker Bot
"""

# Network Architecture
INPUT_SIZE = 186  # State vector size
HIDDEN_LAYERS = [128, 64]
OUTPUT_SIZE = 3  # Number of actions (Check/Bet, Call/Raise, Fold)

# Training Parameters
LEARNING_RATE = 0.001
GAMMA = 0.99  # Discount factor
BATCH_SIZE = 256
TRAIN_ITERATIONS = 500000  # Total training iterations
UPDATE_FREQUENCY = 1  # Update after each step
EVAL_FREQUENCY = 10000  # Report every N iterations
TARGET_UPDATE_FREQ = 500  # Sync target network every N steps

# NFSP Parameters
ANTICIPATORY_PARAM = 0.05  # η (eta) - probability of using best response
TARGET_UPDATE_TAU = 0.001  # Soft update rate for target network

# Memory Parameters
RL_BUFFER_SIZE = 200000
RESERVOIR_SIZE = 2000000
MIN_RL_SAMPLES = 1000  # Minimum samples before training RL
MIN_SL_SAMPLES = 5000  # Minimum samples before training SL

# Evaluation Parameters
EVAL_GAMES = 1000

# Game Parameters
NUM_PLAYERS = 2
DECK_SIZE = 20
STARTING_POT = 2
BET_SIZE = 1

# Device - Auto-detect
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Random Seed
RANDOM_SEED = 42

## 3. Game Environment

In [ ]:
"""
environment.py - Short Deck Poker Environment
20 cards: 10, J, Q, K, A x 4 suits
4 betting rounds: preflop, flop, turn, river
"""

class ShortDeckPokerEnv:
    """
    Short Deck Texas Hold'em Environment for NFSP
    """

    def __init__(self):
        self.num_players = 2
        self.deck_size = 20
        self.hole_cards = 2
        self.community_cards = 5

        # Game state
        self.deck = []
        self.hands = {0: [], 1: []}
        self.board = []
        self.pot = 2
        self.round = 0
        self.current_player = 0
        self.starting_player = 0

        # Betting state
        self.bet_size = 0
        self.player_bets = {0: 0, 1: 0}
        self.actions_this_round = []

        # Terminal state
        self.done = False
        self.winner = None
        self.rewards = {0: 0, 1: 0}

    def reset(self, starting_player=0):
        """Reset game to initial state"""
        self.deck = list(range(20))
        random.shuffle(self.deck)

        # Deal hole cards
        self.hands[0] = [self.deck.pop(), self.deck.pop()]
        self.hands[1] = [self.deck.pop(), self.deck.pop()]

        # Reset game state
        self.board = []
        self.pot = 2
        self.round = 0
        self.starting_player = starting_player
        self.current_player = starting_player

        # Reset betting
        self.bet_size = 0
        self.player_bets = {0: 0, 1: 0}
        self.actions_this_round = []

        # Reset terminal
        self.done = False
        self.winner = None
        self.rewards = {0: 0, 1: 0}

        return self.get_state(0)

    def get_state(self, player):
        """Get 186-bit state vector for player"""
        state = np.zeros(186, dtype=np.float32)

        # Player's hole cards (one-hot encoding)
        for i, card in enumerate(self.hands[player]):
            state[card] = 1.0

        # Community cards (one-hot encoding)
        for i, card in enumerate(self.board):
            state[80 + card] = 1.0

        # Additional features
        state[180] = min(self.pot, 20) / 20.0
        state[181] = self.round / 4.0
        state[182] = 1.0 if self.current_player == player else 0.0
        state[183] = 1.0 if self.bet_size > 0 else 0.0
        state[184] = self.bet_size / 5.0
        state[185] = self.player_bets[player] / 5.0

        return state

    def get_valid_actions(self):
        """Get valid actions for current player"""
        if self.done:
            return []
        if self.bet_size == 0:
            return [0, 1]  # Check or Bet
        else:
            return [0, 1, 2]  # Call, Raise, or Fold

    def step(self, action):
        """Execute action for current player"""
        if self.done:
            raise RuntimeError("Game is over. Call reset() first.")

        valid = self.get_valid_actions()
        if action not in valid:
            raise ValueError(f"Invalid action {action}. Valid: {valid}")

        player = self.current_player
        self.actions_this_round.append((player, action))
        reward = 0

        # Handle Fold
        if action == 2:
            self.done = True
            self.winner = 1 - player
            self.rewards[player] = -self.pot
            self.rewards[1 - player] = self.pot
            reward = self.rewards[0]
            return self.get_state(0), reward, True, self._get_info()

        # Handle Bet/Check
        if self.bet_size == 0:
            if action == 0:  # Check
                if len(self.actions_this_round) == 2:
                    self._advance_round()
                else:
                    self.current_player = 1 - self.current_player
            else:  # Bet
                self.bet_size = 1
                self.player_bets[player] = 1
                self.pot += 1
                self.current_player = 1 - self.current_player
        else:
            # Facing bet
            if action == 0:  # Call
                to_call = self.bet_size - self.player_bets[player]
                self.player_bets[player] = self.bet_size
                self.pot += to_call
                self.bet_size = 0
                self._advance_round()
            else:  # Raise
                self.bet_size += 1
                self.player_bets[player] = self.bet_size
                self.pot += 1
                self.current_player = 1 - self.current_player

        return self.get_state(0), reward, self.done, self._get_info()

    def _advance_round(self):
        """Advance to next round or showdown"""
        self.actions_this_round = []
        self.player_bets = {0: 0, 1: 0}

        if self.round == 0:
            for _ in range(3):
                self.board.append(self.deck.pop())
            self.round = 1
        elif self.round == 1:
            self.board.append(self.deck.pop())
            self.round = 2
        elif self.round == 2:
            self.board.append(self.deck.pop())
            self.round = 3
        elif self.round == 3:
            self.round = 4
            self._showdown()

    def _showdown(self):
        """Compare hands at showdown"""
        self.done = True
        winner = compare_hands(
            self.hands[0] + self.board,
            self.hands[1] + self.board
        )
        self.winner = winner

        if winner == 1:
            self.rewards = {0: -self.pot, 1: self.pot}
        elif winner == -1:
            self.rewards = {0: self.pot, 1: -self.pot}
        else:
            self.rewards = {0: 0, 1: 0}

    def get_reward(self, player):
        """Get reward for player"""
        return self.rewards.get(player, 0)

    def _get_info(self):
        """Get info dict"""
        return {
            "round": self.round,
            "pot": self.pot,
            "current_player": self.current_player,
            "board": self.board,
            "hands": {p: self.hands[p] for p in [0, 1]},
            "winner": self.winner,
            "done": self.done
        }

# Hand evaluation functions
HIGH_CARD = 0
ONE_PAIR = 1
TWO_PAIR = 2
THREE_OF_A_KIND = 3
STRAIGHT = 4
FLUSH = 5
FULL_HOUSE = 6
FOUR_OF_A_KIND = 7
STRAIGHT_FLUSH = 8

def get_rank_counts(cards):
    """Get count of cards for each rank"""
    counts = [0] * 5
    for card in cards:
        if 0 <= card < 20:
            rank = card // 4
            counts[rank] += 1
    return counts

def get_suit_counts(cards):
    """Get count of cards for each suit"""
    counts = [0] * 4
    for card in cards:
        if 0 <= card < 20:
            suit = card % 4
            counts[suit] += 1
    return counts

def is_flush(cards):
    """Check if 5+ cards form a flush"""
    return any(s >= 5 for s in get_suit_counts(cards))

def is_straight(cards):
    """Check if cards contain a straight"""
    rank_mask = 0
    for card in cards:
        if 0 <= card < 20:
            rank = card // 4
            rank_mask |= 1 << rank

    # Check for 5 consecutive bits (short deck: only 10-J-Q-K-A)
    for i in range(5):
        needed = 0b11111 << i
        if (rank_mask & needed) == needed:
            return True
    return False

def evaluate_hand(cards):
    """Evaluate best 5-card hand from 7 cards"""
    if len(cards) < 5:
        return (HIGH_CARD, ())

    best = None
    for combo in combinations(cards, 5):
        hand = evaluate_5_cards(list(combo))
        if best is None or hand > best:
            best = hand

    return best

def evaluate_5_cards(cards):
    """Evaluate a 5-card hand"""
    rank_counts = get_rank_counts(cards)
    has_flush = is_flush(cards)
    has_straight = is_straight(cards)

    # Straight flush
    if has_flush and has_straight:
        return (STRAIGHT_FLUSH, ())

    # Four of a kind
    if 4 in rank_counts:
        kicker = [r for r in range(4, -1, -1) if rank_counts[r] != 4][0]
        return (FOUR_OF_A_KIND, (rank_counts.index(4), kicker))

    # Full house
    if 3 in rank_counts and 2 in rank_counts:
        trips = rank_counts.index(3)
        pair = [r for r in range(4, -1, -1) if rank_counts[r] == 2][0]
        return (FULL_HOUSE, (trips, pair))

    # Flush
    if has_flush:
        kickers = sorted([c // 4 for c in cards], reverse=True)[:5]
        return (FLUSH, tuple(kickers))

    # Straight
    if has_straight:
        return (STRAIGHT, ())

    # Three of a kind
    if 3 in rank_counts:
        trips = rank_counts.index(3)
        kickers = tuple(r for r in range(4, -1, -1) if r != trips)
        return (THREE_OF_A_KIND, kickers)

    # Two pair
    pairs = sorted([r for r in range(5) if rank_counts[r] == 2], reverse=True)
    if len(pairs) >= 2:
        kicker = [r for r in range(4, -1, -1) if r not in pairs][0]
        return (TWO_PAIR, (pairs[0], pairs[1], kicker))

    # One pair
    if 2 in rank_counts:
        pair = rank_counts.index(2)
        kickers = tuple(r for r in range(4, -1, -1) if r != pair)
        return (ONE_PAIR, kickers)

    # High card
    kickers = tuple(sorted([c // 4 for c in cards], reverse=True))
    return (HIGH_CARD, kickers)

def compare_hands(cards1, cards2):
    """Compare two hands"""
    hand1 = evaluate_hand(cards1)
    hand2 = evaluate_hand(cards2)

    if hand1 > hand2:
        return -1  # Player 0 wins
    elif hand2 > hand1:
        return 1   # Player 1 wins
    else:
        return 0   # Tie

def card_to_string(card):
    """Convert card index to string"""
    ranks = ['10', 'J', 'Q', 'K', 'A']
    suits = ['♣', '♦', '♥', '♠']
    return ranks[card // 4] + suits[card % 4]

print("Environment loaded successfully!")

## 4. Neural Networks

In [ ]:
"""
networks.py - Neural Network Architectures
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class QNetwork(nn.Module):
    """Q-Network for RL (Best Response)"""
    
    def __init__(self, input_size, hidden_layers, output_size):
        super().__init__()
        
        # Build layers dynamically
        layers = []
        prev_size = input_size
        for hidden_size in hidden_layers:
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.ReLU()
            ])
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, output_size))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)


class AveragePolicyNetwork(nn.Module):
    """Average Policy Network for SL (Average Strategy)"""
    
    def __init__(self, input_size, hidden_layers, output_size):
        super().__init__()
        
        layers = []
        prev_size = input_size
        for hidden_size in hidden_layers:
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.ReLU()
            ])
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, output_size))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return F.softmax(self.network(x), dim=-1)


# Using QNetwork directly for target network (same architecture)
TargetNetwork = QNetwork


def create_networks(input_size=INPUT_SIZE, hidden_layers=HIDDEN_LAYERS, output_size=OUTPUT_SIZE):
    """Create all networks"""
    q_net = QNetwork(input_size, hidden_layers, output_size)
    target_net = TargetNetwork(input_size, hidden_layers, output_size)
    avg_policy_net = AveragePolicyNetwork(input_size, hidden_layers, output_size)
    
    return q_net, target_net, avg_policy_net

print("Networks loaded successfully!")

## 5. Memory Modules

In [ ]:
"""
replay_buffer.py & reservoir.py - Memory Stores
"""

class ReplayBuffer:
    """Circular buffer for RL experiences"""
    
    def __init__(self, capacity):
        self.buffer = [None] * capacity
        self.capacity = capacity
        self.position = 0
        self.size = 0
    
    def push(self, state, action, reward, next_state, done):
        """Add experience"""
        self.buffer[self.position] = (state, action, reward, next_state, done)
        self.position = (self.position + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)
    
    def sample(self, batch_size):
        """Sample random batch"""
        indices = random.sample(range(self.size), min(batch_size, self.size))
        states, actions, rewards, next_states, dones = zip(*[self.buffer[i] for i in indices])
        
        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32)
        )
    
    def __len__(self):
        return self.size


class ReservoirSampling:
    """Reservoir sampling for SL experiences"""
    
    def __init__(self, capacity):
        self.reservoir = []
        self.capacity = capacity
        self.count = 0
    
    def push(self, state, action):
        """Add experience with reservoir sampling"""
        self.count += 1
        
        if len(self.reservoir) < self.capacity:
            self.reservoir.append((state, action))
        else:
            # Reservoir sampling
            j = random.randint(0, self.count - 1)
            if j < self.capacity:
                self.reservoir[j] = (state, action)
    
    def sample(self, batch_size):
        """Sample random batch"""
        if len(self.reservoir) == 0:
            return None, None
        
        indices = random.sample(range(len(self.reservoir)), min(batch_size, len(self.reservoir)))
        states, actions = zip(*[self.reservoir[i] for i in indices])
        
        return np.array(states, dtype=np.float32), np.array(actions, dtype=np.int64)
    
    def __len__(self):
        return len(self.reservoir)

print("Memory modules loaded successfully!")

## 6. NFSP Agent

In [ ]:
"""
nfsp_agent.py - NFSP Agent Implementation
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random


class NSFPAgent:
    """
    Neural Fictitious Self-Play Agent
    
    Combines:
    - RL (DQN) for best response
    - SL (Supervised Learning) for average policy
    """
    
    def __init__(self, device=DEVICE):
        self.device = torch.device(device)
        
        # Networks
        self.q_net = QNetwork(INPUT_SIZE, HIDDEN_LAYERS, OUTPUT_SIZE).to(self.device)
        self.target_net = TargetNetwork(INPUT_SIZE, HIDDEN_LAYERS, OUTPUT_SIZE).to(self.device)
        self.avg_policy_net = AveragePolicyNetwork(INPUT_SIZE, HIDDEN_LAYERS, OUTPUT_SIZE).to(self.device)
        
        # Copy weights to target network
        self.target_net.load_state_dict(self.q_net.state_dict())
        
        # Optimizers
        self.q_optimizer = torch.optim.Adam(self.q_net.parameters(), lr=LEARNING_RATE)
        self.sl_optimizer = torch.optim.Adam(self.avg_policy_net.parameters(), lr=LEARNING_RATE)
        
        # Memory
        self.rl_buffer = ReplayBuffer(RL_BUFFER_SIZE)
        self.sl_reservoir = ReservoirSampling(RESERVOIR_SIZE)
        
        # Statistics
        self.total_steps = 0
        self.rl_updates = 0
        self.sl_updates = 0
        
        # Exploration flag
        self.use_best_response = False
    
    def choose_action(self, state, valid_actions, epsilon=None):
        """Choose action using mixture policy"""
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        
        # With probability eta, use best response (greedy Q)
        if self.use_best_response or random.random() < ANTICIPATORY_PARAM:
            # Best response (greedy)
            with torch.no_grad():
                q_values = self.q_net(state_tensor).squeeze()
            
            # Mask invalid actions
            masked_q = torch.full((OUTPUT_SIZE,), float('-inf'))
            for action in valid_actions:
                masked_q[action] = q_values[action]
            
            action = masked_q.argmax().item()
        else:
            # Average policy (stochastic)
            with torch.no_grad():
                probs = self.avg_policy_net(state_tensor).squeeze()
            
            # Mask invalid actions
            masked_probs = torch.zeros(OUTPUT_SIZE)
            for action in valid_actions:
                masked_probs[action] = probs[action]
            
            # Re-normalize
            masked_probs = masked_probs / masked_probs.sum()
            
            action = torch.multinomial(masked_probs, 1).item()
        
        return action
    
    def store_experience(self, state, action, reward, next_state, done, valid_actions):
        """Store experience in both memories"""
        # RL buffer stores full transitions
        self.rl_buffer.push(state, action, reward, next_state, done)
        
        # SL reservoir stores only (state, action) pairs
        # Only when using best response
        if self.use_best_response or random.random() < ANTICIPATORY_PARAM:
            self.sl_reservoir.push(state, action)
        
        self.total_steps += 1
    
    def update_rl(self):
        """Update Q-network using DQN"""
        if len(self.rl_buffer) < MIN_RL_SAMPLES:
            return None
        
        # Sample batch
        states, actions, rewards, next_states, dones = self.rl_buffer.sample(BATCH_SIZE)
        
        # Convert to tensors
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)
        
        # Compute current Q values
        q_values = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze()
        
        # Compute target Q values
        with torch.no_grad():
            next_q_values = self.target_net(next_states).max(1)[0]
            target_q = rewards + (1 - dones) * GAMMA * next_q_values
        
        # Compute loss and update
        loss = nn.MSELoss()(q_values, target_q)
        self.q_optimizer.zero_grad()
        loss.backward()
        self.q_optimizer.step()
        
        self.rl_updates += 1
        return loss.item()
    
    def update_sl(self):
        """Update average policy using supervised learning"""
        if len(self.sl_reservoir) < MIN_SL_SAMPLES:
            return None
        
        # Sample batch
        states, actions = self.sl_reservoir.sample(BATCH_SIZE)
        
        if states is None:
            return None
        
        # Convert to tensors
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        
        # Compute predicted policy
        predicted_probs = self.avg_policy_net(states)
        
        # Cross-entropy loss
        loss = F.cross_entropy(predicted_probs, actions)
        
        self.sl_optimizer.zero_grad()
        loss.backward()
        self.sl_optimizer.step()
        
        self.sl_updates += 1
        return loss.item()
    
    def sync_target_network(self):
        """Soft update target network"""
        for target_param, param in zip(
            self.target_net.parameters(), 
            self.q_net.parameters()
        ):
            target_param.data.copy_(
                TARGET_UPDATE_TAU * param.data + 
                (1 - TARGET_UPDATE_TAU) * target_param.data
            )
    
    def get_statistics(self):
        """Get agent statistics"""
        return {
            'total_steps': self.total_steps,
            'rl_buffer_size': len(self.rl_buffer),
            'sl_reservoir_size': len(self.sl_reservoir),
            'rl_updates': self.rl_updates,
            'sl_updates': self.sl_updates,
        }
    
    def save(self, path):
        """Save model"""
        torch.save({
            'q_net': self.q_net.state_dict(),
            'target_net': self.target_net.state_dict(),
            'avg_policy_net': self.avg_policy_net.state_dict(),
            'q_optimizer': self.q_optimizer.state_dict(),
            'sl_optimizer': self.sl_optimizer.state_dict(),
            'stats': self.get_statistics()
        }, path)
        print(f"Model saved to {path}")
    
    def load(self, path):
        """Load model"""
        checkpoint = torch.load(path, map_location=self.device)
        self.q_net.load_state_dict(checkpoint['q_net'])
        self.target_net.load_state_dict(checkpoint['target_net'])
        self.avg_policy_net.load_state_dict(checkpoint['avg_policy_net'])
        self.q_optimizer.load_state_dict(checkpoint['q_optimizer'])
        self.sl_optimizer.load_state_dict(checkpoint['sl_optimizer'])
        print(f"Model loaded from {path}")


# Evaluation opponents
def get_random_opponent_action(valid_actions):
    """Random baseline"""
    return random.choice(valid_actions)


def get_heuristic_opponent(env, valid_actions):
    """Hand-strength based opponent"""
    player = env.current_player
    hand = env.hands[player] + env.board
    hand_rank = evaluate_hand(hand)[0]
    
    # Play tight: only bet/raise with strong hands
    if hand_rank >= ONE_PAIR:
        if 1 in valid_actions:  # Bet/Raise available
            return 1
    
    # Call or check
    if 0 in valid_actions:
        return 0
    
    return valid_actions[-1]


def evaluate_agent(agent, opponent_type, n_games=1000):
    """Evaluate agent against baseline opponent"""
    wins = 0
    
    for _ in range(n_games):
        env = ShortDeckPokerEnv()
        state = env.reset()
        
        while not env.done:
            valid_actions = env.get_valid_actions()
            
            if env.current_player == 0:
                # Our agent
                agent.use_best_response = True  # Pure greedy
                action = agent.choose_action(state, valid_actions)
            else:
                # Opponent
                if opponent_type == 'random':
                    action = get_random_opponent_action(valid_actions)
                elif opponent_type == 'heuristic':
                    action = get_heuristic_opponent(env, valid_actions)
                else:
                    action = get_random_opponent_action(valid_actions)
            
            state, reward, done, info = env.step(action)
        
        # Player 0 reward (our agent)
        if env.rewards[0] > 0:
            wins += 1
        elif env.rewards[0] == 0:
            wins += 0.5  # Tie
    
    return wins / n_games

print("NFSP Agent loaded successfully!")

## 7. Training Function

In [ ]:
def set_seed(seed):
    """Set random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)


def play_episode(env, agent1, agent2, store_experiences=True):
    """Play one episode of self-play between two agents"""
    state = env.reset(starting_player=random.randint(0, 1))
    
    episode_stats = {
        'bets': 0,
        'folds': 0,
        'pot_size': 0,
        'episode_length': 0
    }
    
    while not env.done:
        current_player = env.current_player
        agent = agent1 if current_player == 0 else agent2
        
        valid_actions = env.get_valid_actions()
        action = agent.choose_action(state, valid_actions)
        
        if action not in valid_actions:
            action = valid_actions[0]
        
        episode_stats['episode_length'] += 1
        if action in [1, 2]:  # Bet/Raise or Fold
            if action == 2:
                episode_stats['folds'] += 1
            else:
                episode_stats['bets'] += 1
        
        next_state, reward, done, info = env.step(action)
        
        if store_experiences:
            if current_player == 0:
                agent1.store_experience(state, action, reward, next_state, done, valid_actions)
            else:
                agent2.store_experience(state, action, -reward, next_state, done, valid_actions)
        
        state = next_state
    
    episode_stats['pot_size'] = env.pot
    episode_stats['winner'] = env.winner
    
    return env.winner, episode_stats


def update_agents(agent1, agent2, force_update=False):
    """Update both agents"""
    rl_losses_1, sl_losses_1 = [], []
    rl_losses_2, sl_losses_2 = [], []
    
    if force_update or (len(agent1.rl_buffer) >= MIN_RL_SAMPLES and len(agent1.sl_reservoir) >= MIN_SL_SAMPLES):
        rl_loss_1 = agent1.update_rl()
        sl_loss_1 = agent1.update_sl()
        if rl_loss_1 is not None:
            rl_losses_1.append(rl_loss_1)
        if sl_loss_1 is not None:
            sl_losses_1.append(sl_loss_1)
        
        rl_loss_2 = agent2.update_rl()
        sl_loss_2 = agent2.update_sl()
        if rl_loss_2 is not None:
            rl_losses_2.append(rl_loss_2)
        if sl_loss_2 is not None:
            sl_losses_2.append(sl_loss_2)
    
    return {
        'agent1_rl_loss': np.mean(rl_losses_1) if rl_losses_1 else None,
        'agent1_sl_loss': np.mean(sl_losses_1) if sl_losses_1 else None,
        'agent2_rl_loss': np.mean(rl_losses_2) if rl_losses_2 else None,
        'agent2_sl_loss': np.mean(sl_losses_2) if sl_losses_2 else None,
    }


def compute_metrics(episode_stats_list):
    """Compute average metrics from episode history"""
    if not episode_stats_list:
        return {}
    
    total_bets = sum(s['bets'] for s in episode_stats_list)
    total_folds = sum(s['folds'] for s in episode_stats_list)
    total_actions = sum(s['episode_length'] for s in episode_stats_list)
    
    return {
        'avg_pot': np.mean([s['pot_size'] for s in episode_stats_list]),
        'avg_length': np.mean([s['episode_length'] for s in episode_stats_list]),
        'bet_rate': total_bets / max(1, total_actions),
        'fold_rate': total_folds / max(1, total_actions),
    }

print("Training functions loaded successfully!")

## 8. Main Training Loop

In [ ]:
def train(n_iterations=TRAIN_ITERATIONS, save_every=50000, verbose=True):
    """Main training loop with visualization"""
    
    print("=" * 60)
    print("NFSP Poker Bot Training")
    print("=" * 60)
    print(f"\nDevice: {DEVICE}")
    print(f"Training iterations: {n_iterations:,}")
    print(f"Save every: {save_every:,}")
    
    # Set seed
    set_seed(RANDOM_SEED)
    
    # Create environment and agents
    env = ShortDeckPokerEnv()
    agent1 = NSFPAgent(device=DEVICE)
    agent2 = NSFPAgent(device=DEVICE)
    
    # Training stats
    recent_episodes = deque(maxlen=1000)
    recent_wins = deque(maxlen=1000)
    
    # Metrics history for plotting
    metrics_history = {
        'iterations': [],
        'rl_loss': [],
        'sl_loss': [],
        'win_rate': [],
        'bet_rate': [],
        'fold_rate': [],
    }
    
    start_time = time.time()
    steps_since_sync = 0
    
    for iteration in range(n_iterations):
        # Play episode
        winner, episode_stats = play_episode(env, agent1, agent2)
        recent_episodes.append(episode_stats)
        
        if winner == 0:
            recent_wins.append(1)
        elif winner == 1:
            recent_wins.append(0)
        else:
            recent_wins.append(0.5)
        
        # Update agents
        update_stats = update_agents(agent1, agent2)
        
        # Sync target networks periodically
        steps_since_sync += 1
        if steps_since_sync >= TARGET_UPDATE_FREQ:
            agent1.sync_target_network()
            agent2.sync_target_network()
            steps_since_sync = 0
        
        # Periodic logging and visualization
        if (iteration + 1) % save_every == 0:
            elapsed = time.time() - start_time
            metrics = compute_metrics(list(recent_episodes))
            
            # Compute win rate
            win_rate = sum(recent_wins) / len(recent_wins)
            
            # Record metrics
            metrics_history['iterations'].append(iteration + 1)
            metrics_history['win_rate'].append(win_rate)
            metrics_history['bet_rate'].append(metrics.get('bet_rate', 0))
            metrics_history['fold_rate'].append(metrics.get('fold_rate', 0))
            
            if update_stats['agent1_rl_loss']:
                metrics_history['rl_loss'].append(update_stats['agent1_rl_loss'])
            if update_stats['agent1_sl_loss']:
                metrics_history['sl_loss'].append(update_stats['agent1_sl_loss'])
            
            # Clear output and plot (optimized)
            clear_output(wait=True)
            
            # Limit history size for faster plotting
            max_plot_points = 50
            plot_indices = list(range(0, len(metrics_history['iterations']), 
                                      max(1, len(metrics_history['iterations']) // max_plot_points)))
            
            # Create figure
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            
            iters = [metrics_history['iterations'][i] for i in plot_indices]
            
            # Win rate
            wr = [metrics_history['win_rate'][i] for i in plot_indices]
            axes[0, 0].plot(iters, wr, 'b-', linewidth=2)
            axes[0, 0].axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
            axes[0, 0].set_title('Self-Play Win Rate', fontsize=14)
            axes[0, 0].set_xlabel('Iterations')
            axes[0, 0].set_ylabel('Win Rate')
            axes[0, 0].set_ylim([0, 1])
            axes[0, 0].grid(True, alpha=0.3)
            
            # Bet/Fold rate
            br = [metrics_history['bet_rate'][i] for i in plot_indices]
            fr = [metrics_history['fold_rate'][i] for i in plot_indices]
            axes[0, 1].plot(iters, br, 'g-', linewidth=2, label='Bet')
            axes[0, 1].plot(iters, fr, 'r-', linewidth=2, label='Fold')
            axes[0, 1].set_title('Action Rates', fontsize=14)
            axes[0, 1].set_xlabel('Iterations')
            axes[0, 1].set_ylabel('Rate')
            axes[0, 1].legend()
            axes[0, 1].grid(True, alpha=0.3)
            
            # RL Loss (last values only)
            if metrics_history['rl_loss']:
                rl_plot = metrics_history['rl_loss'][-max_plot_points:]
                rl_iters = metrics_history['iterations'][-len(rl_plot):]
                axes[1, 0].plot(rl_iters, rl_plot, 'orange', linewidth=2)
            axes[1, 0].set_title('RL Loss (DQN)', fontsize=14)
            axes[1, 0].set_xlabel('Iterations')
            axes[1, 0].set_ylabel('Loss')
            axes[1, 0].grid(True, alpha=0.3)
            
            # SL Loss (last values only)
            if metrics_history['sl_loss']:
                sl_plot = metrics_history['sl_loss'][-max_plot_points:]
                sl_iters = metrics_history['iterations'][-len(sl_plot):]
                axes[1, 1].plot(sl_iters, sl_plot, 'purple', linewidth=2)
            axes[1, 1].set_title('SL Loss (Cross-Entropy)', fontsize=14)
            axes[1, 1].set_xlabel('Iterations')
            axes[1, 1].set_ylabel('Loss')
            axes[1, 1].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
            # Print stats
            print(f"\n{'='*60}")
            print(f"Iteration: {iteration + 1:,} / {n_iterations:,} ({100*(iteration+1)/n_iterations:.1f}%)")
            print(f"Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")
            print(f"Speed: {(iteration+1)/elapsed:.0f} it/s")
            print(f"{'='*60}")
            print(f"Win Rate: {win_rate:.2%}")
            print(f"Avg Pot: {metrics.get('avg_pot', 0):.2f}")
            print(f"Bet Rate: {metrics.get('bet_rate', 0):.2%}")
            print(f"Fold Rate: {metrics.get('fold_rate', 0):.2%}")
            print(f"{'='*60}")
            print(f"RL Buffer: {len(agent1.rl_buffer):,}")
            print(f"SL Reservoir: {len(agent1.sl_reservoir):,}")
            if update_stats['agent1_rl_loss']:
                print(f"RL Loss: {update_stats['agent1_rl_loss']:.4f}")
            if update_stats['agent1_sl_loss']:
                print(f"SL Loss: {update_stats['agent1_sl_loss']:.4f}")
            
            # Save checkpoint
            checkpoint_path = f"checkpoint_{iteration + 1}.pt"
            agent1.save(checkpoint_path)
            print(f"\nCheckpoint saved: {checkpoint_path}")
    
    # Save final models
    print("\n" + "=" * 60)
    print("Training Complete!")
    print("=" * 60)
    
    elapsed = time.time() - start_time
    print(f"Total time: {elapsed:.1f}s ({elapsed/3600:.2f} hours)")
    
    agent1.save("nfsp_agent_final.pt")
    agent2.save("nfsp_agent2_final.pt")
    
    return agent1, agent2, metrics_history

print("Training loop loaded successfully!")

## 9. Start Training!

In [ ]:
# Start training
# Adjust iterations as needed - 500k is a good default for convergence

# For quick test, use fewer iterations:
# agent1, agent2, history = train(n_iterations=10000, save_every=2000)

# For full training:
agent1, agent2, history = train(n_iterations=500000, save_every=10000)

## 10. Evaluate Trained Agent

In [ ]:
# Load the best checkpoint
best_agent = NSFPAgent(device=DEVICE)
best_agent.load("nfsp_agent_final.pt")

# Evaluate against different opponents
print("\n" + "=" * 60)
print("Evaluation Results")
print("=" * 60)

opponents = ['random', 'heuristic']
n_eval_games = 2000

for opponent in opponents:
    win_rate = evaluate_agent(best_agent, opponent, n_games=n_eval_games)
    print(f"vs {opponent.capitalize():12s}: {win_rate:.2%} ({n_eval_games} games)")

print("\nNote: >50% means the agent is outperforming the baseline")

## 11. Play Against the Agent

In [ ]:
def print_game_state(env):
    """Display current game state"""
    print("\n" + "=" * 40)
    print(f"Round: {['Pre-flop', 'Flop', 'Turn', 'River', 'Showdown'][env.round]}")
    print(f"Pot: {env.pot}")
    print(f"Board: {' '.join([card_to_string(c) for c in env.board])}")
    print("-" * 40)
    
    if not env.done:
        print(f"Your hand: {' '.join([card_to_string(c) for c in env.hands[1]])}")
        valid = env.get_valid_actions()
        actions = ['Check', 'Bet', 'Fold']
        print(f"Valid actions: {[actions[a] for a in valid]}")
    else:
        print(f"Your hand: {' '.join([card_to_string(c) for c in env.hands[1]])}")
        print(f"AI hand: {' '.join([card_to_string(c) for c in env.hands[0]])}")
        if env.winner == 1:
            print(">>> YOU WIN! <<<")
        elif env.winner == 0:
            print(">>> AI WINS <<<")
        else:
            print(">>> TIE <<<")
    print("=" * 40)


def play_against_agent(agent, n_hands=5):
    """Play hands against the trained agent"""
    agent.use_best_response = True  # Use greedy policy
    
    total_wins = 0
    
    for hand_num in range(1, n_hands + 1):
        print(f"\n{'#'*50}")
        print(f"HAND #{hand_num}")
        print(f"{'#'*50}")
        
        env = ShortDeckPokerEnv()
        state = env.reset(starting_player=1)  # You are player 1
        
        while not env.done:
            print_game_state(env)
            
            valid_actions = env.get_valid_actions()
            
            if env.current_player == 1:  # Your turn
                print("\nEnter your action:")
                print("  0 = Check/Call")
                print("  1 = Bet/Raise")
                print("  2 = Fold")
                
                while True:
                    try:
                        action = int(input("Action (0/1/2): "))
                        if action in valid_actions:
                            break
                        print(f"Invalid action. Valid: {valid_actions}")
                    except:
                        print("Enter 0, 1, or 2")
            else:  # AI's turn
                action = agent.choose_action(state, valid_actions)
                print(f"AI chooses: {['Check', 'Bet', 'Fold'][action]}")
            
            state, reward, done, info = env.step(action)
        
        # Showdown
        print_game_state(env)
        
        if env.rewards[1] > 0:
            total_wins += 1
    
    print(f"\n{'='*50}")
    print(f"Results: {total_wins}/{n_hands} wins ({100*total_wins/n_hands:.0f}%)")
    print(f"{'='*50}")

# Uncomment to play:
# play_against_agent(best_agent, n_hands=5)

---

## Tips for Colab Training

1. **Runtime → Change runtime type → GPU** for faster training
2. **Save checkpoints to Google Drive** for persistence
3. **Monitor GPU usage** with `!nvidia-smi`
4. **Training 500k iterations** typically takes 4-8 hours on GPU
5. **Connect to Google Drive** to save your trained model:
```python
from google.colab import drive
drive.mount('/content/gdrive')
# Then save to /content/gdrive/MyDrive/
```

## Expected Training Timeline

- **10k iterations**: Agent learns basic betting
- **50k iterations**: Agent develops consistent strategy
- **200k iterations**: Agent becomes competitive (>55% vs heuristic)
- **500k iterations**: Near-equilibrium play (60%+ vs heuristic)